# Precompute Qwen Knowledge for FCF Distillation

Extracts pairwise token similarities from Qwen3-4B over the FCF training corpus.

**Platforms:** Google Colab (T4/V100/A100) • Yandex DataSphere (V100)

**Resume support:** if the session times out, re-run cell 6 to continue from checkpoint.

**Setup:**
1. Runtime → Change runtime type → GPU
2. Run cells in order — files auto-download from shared Drive folder

**Output:** `qwen_knowledge.npz` (~200-500 MB, saved to Drive on Colab, working dir on DataSphere)

**Estimated runtime:** 1-6 hours (A100 ~1h, V100 ~3h, T4 ~6h)

## 1. Install dependencies

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_GPU'))

reqs = [
    "gdown",  # download files from shared Drive
    "sentencepiece",
    "accelerate>=0.26.0",
    "transformers>=4.45.0",
]
if not IN_COLAB:
    # DataSphere has torch 2.0.1 — upgrade for transformers 4.45+
    reqs.insert(0, "torch>=2.1.0")

for pkg in reqs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"Dependencies installed  |  torch {torch.__version__}, CUDA {torch.cuda.is_available()}")

## 2. Download input files

Downloads corpus, BPE model and precompute script from a shared Google Drive folder.
Requires the folder to be publicly accessible (Anyone with the link → Viewer).

In [ ]:
import gdown, shutil

DATA_DIR = os.getcwd()
print(f"Working directory: {DATA_DIR}")

SHARED_FOLDER_ID = "1L8T0fDK7QlCQSFsE3fUCai5sNIFCHEdn"
print("Downloading files from shared Drive folder...")
gdown.download_folder(f"https://drive.google.com/drive/folders/{SHARED_FOLDER_ID}",
                     quiet=False, output=DATA_DIR)

# Platform-aware output path
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_GPU'))
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/fcf_data/"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for fname in ["full_corpus_ru_clean.txt", "bpe_ru_146k.model", "precompute_qwen_knowledge.py"]:
        src = os.path.join(DATA_DIR, fname)
        dst = os.path.join(DRIVE_DIR, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    output_path = os.path.join(DRIVE_DIR, "qwen_knowledge.npz")
    print(f"Output → Drive: {output_path}")
else:
    output_path = os.path.join(DATA_DIR, "qwen_knowledge.npz")
    print(f"Output → {output_path}")

# Verify
corpus_path   = os.path.join(DATA_DIR, "full_corpus_ru_clean.txt")
fcf_bpe_path  = os.path.join(DATA_DIR, "bpe_ru_146k.model")
script_path   = os.path.join(DATA_DIR, "precompute_qwen_knowledge.py")

for p, name in [(corpus_path, "corpus"), (fcf_bpe_path, "BPE model"), (script_path, "script")]:
    ok = os.path.exists(p)
    print(f"  {name}: {'OK' if ok else 'MISSING'} ({os.path.getsize(p)/1e6 if ok else 0:.1f} MB)")
    if not ok:
        raise RuntimeError(f"Missing {name} — check shared folder link")

## 3. Load Qwen model (with optimizations)

Downloads from HuggingFace: `RefalMachine/RuadaptQwen3-4B-Hybrid`

Optimizations enabled:
- `flash_attention_2` (2-3x speedup on compatible GPU)
- `torch.float16` (half the memory, same quality for hidden states)
- `torch.inference_mode()` (faster than no_grad)
- **early exit**: configurable `NUM_LAYERS` — fewer layers = faster

In [ ]:
import torch
print(f"  torch {torch.__version__}, CUDA {torch.cuda.is_available()}")
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "RefalMachine/RuadaptQwen3-4B-Hybrid"
NUM_LAYERS = 12  # Use first 12 layers (out of 36). 12 = 3x faster, ~95% quality.
                 # Set to None for all 36 layers (best quality, 3x slower)

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Loading model...")
# flash_attention_2 needs sm75+ (V100=sm70, A100=sm80). Auto-detect.
try:
    import flash_attn  # noqa: F401
    _fa_ok = True
except ImportError:
    _fa_ok = False
attn_impl = "flash_attention_2" if (torch.cuda.is_available() and _fa_ok) else "eager"
print(f"  Attention: {attn_impl}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
    low_cpu_mem_usage=True,
    attn_implementation=attn_impl,
)

# Early exit: truncate to first N layers
if NUM_LAYERS is not None:
    total = len(model.model.layers)
    model.model.layers = model.model.layers[:NUM_LAYERS]
    print(f"  Truncated to first {NUM_LAYERS}/{total} layers")

model.eval()
device = next(model.parameters()).device
dtype = next(model.parameters()).dtype
print(f"  Device: {device}, dtype: {dtype}")
print(f"  Hidden size: {model.config.hidden_size}")

## 4. Load FCF BPE + verify tokenization

In [ ]:
import sentencepiece as spm

# Load FCF BPE
sp = spm.SentencePieceProcessor()
sp.Load(fcf_bpe_path)
print(f"FCF BPE vocab: {sp.vocab_size()}")

# Verify the precompute script is importable
sys.path.insert(0, DATA_DIR)
from precompute_qwen_knowledge import encode_fcf_spans, align_and_similarities, StreamingAccum, merge_partials
print("Precompute module loaded OK")

# Quick alignment test
test_text = "Привет мир! Как дела?"
print(f"\nTest text: '{test_text}'")
cids, spans = encode_fcf_spans(test_text, sp)
enc = tokenizer(test_text, return_offsets_mapping=True, add_special_tokens=False)
print(f"  FCF: {len(cids)} tokens, Qwen: {len(enc['input_ids'])} tokens")
for i, (cid, (s, e)) in enumerate(zip(cids, spans)):
    print(f"    FCF[{i}] cid={cid:5d} '{test_text[s:e]}'")

## 5. Benchmark

Estimates total time from 50 random samples.

In [ ]:
import random, time
import numpy as np

with open(corpus_path, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Corpus: {len(lines)} texts")

max_len = min(model.config.max_position_embeddings, 2048)
print(f"Max seq len: {max_len}")

random.seed(42)
samples = random.sample(lines, min(50, len(lines)))

# Warmup
with torch.inference_mode():
    _ = model(tokenizer(samples[0], return_tensors="pt", truncation=True, max_length=128)["input_ids"].to(device))

# Benchmark
times = []; total_tokens = 0
for text in samples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len)
    n_tok = inputs["input_ids"].shape[1]
    total_tokens += n_tok
    t0 = time.time()
    with torch.inference_mode():
        _ = model(inputs["input_ids"].to(device), output_hidden_states=True)
    times.append(time.time() - t0)

avg_ms = np.mean(times) * 1000
tok_s = total_tokens / sum(times)
hours_est = len(lines) * avg_ms / 1000 / 3600
print(f"\nBenchmark ({len(samples)} texts, {NUM_LAYERS or 36} layers):")
print(f"  Avg: {avg_ms:.0f} ms per text | {tok_s:.0f} tok/s")
print(f"  Estimated: {hours_est:.1f} hours for {len(lines)} texts")
print(f"  (early exit {NUM_LAYERS}/{36} layers = ~{36/(NUM_LAYERS or 36):.1f}x speedup)")

## 6. Run precomputation

This processes the corpus and accumulates pairwise similarities.
Partial files are saved every 1000 texts as safety against crashes.

In [ ]:
import time, gc, glob
import numpy as np

CONTEXT_WINDOW = 8
CHECKPOINT_EVERY = 1000
MAX_LEN = min(model.config.max_position_embeddings, 2048)

# === Resume support ===
progress_path = output_path + ".progress"
start_idx = 0
existing_partials = sorted(glob.glob(output_path + ".part[0-9]*.npz"))

if existing_partials and os.path.exists(progress_path):
    with open(progress_path) as f:
        start_idx = int(f.read().strip())
    print(f"Resuming from text {start_idx}/{len(lines)} ({len(existing_partials)} partials exist)")

# === Main loop ===
accum = StreamingAccum(output_path)
t_start = time.time()
n_pairs_total = 0
n_texts = start_idx

for idx, text in enumerate(lines):
    if idx < start_idx:
        continue
    if not text:
        continue

    # --- Tokenize ---
    cids, fcf_spans = encode_fcf_spans(text, sp)
    enc = tokenizer(text, return_offsets_mapping=True,
                    add_special_tokens=False, truncation=True, max_length=MAX_LEN)
    qw_ids = enc["input_ids"]
    qw_spans = enc["offset_mapping"]

    if not qw_ids or not cids:
        continue

    # --- Qwen forward pass ---
    with torch.inference_mode():
        inputs = torch.tensor([qw_ids], device=device)
        outputs = model(inputs, output_hidden_states=True)
        qw_hidden = outputs.hidden_states[-1][0].cpu().numpy()

    if len(qw_hidden) != len(qw_ids):
        continue

    # --- Align + pairwise similarities ---
    results = align_and_similarities(text, cids, fcf_spans, qw_hidden, qw_spans, CONTEXT_WINDOW)

    if results:
        accum.add(results)
        n_pairs_total += len(results)
    n_texts += 1

    # --- Periodic flush ---
    if accum.should_flush():
        accum.flush()

    if (idx + 1) % CHECKPOINT_EVERY == 0:
        # Save progress for resume
        with open(progress_path, "w") as f:
            f.write(str(idx + 1))

        elapsed = time.time() - t_start
        processed = idx + 1 - start_idx
        rate = processed / elapsed if elapsed > 0 else 0
        remaining = len(lines) - idx - 1
        eta = remaining / rate if rate > 0 else 0
        gpu_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        print(f"  [{100*(idx+1)/len(lines):4.1f}%] {idx+1}/{len(lines)} "
              f"| {n_pairs_total} pairs | {rate:.0f} L/s "
              f"| ETA {eta/3600:.1f}h | GPU {gpu_mem:.1f}GB")

    if (idx + 1) % 200 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# --- Flush + merge ---
if accum.data:
    accum.flush()

all_partials = sorted(glob.glob(output_path + ".part[0-9]*.npz"))
if all_partials:
    print(f"Merging {len(all_partials)} partial files...")
    merge_partials(all_partials, output_path)
    if os.path.exists(progress_path):
        os.remove(progress_path)

total = time.time() - t_start
print(f"\nDONE: {n_texts} texts, {n_pairs_total} pairs in {total/3600:.2f}h")
print(f"  Avg: {n_texts/total:.0f} L/s | {n_pairs_total/total:.0f} pairs/s")

## 7. Verify output

In [ ]:
data = np.load(output_path)
n = len(data["rows"])
vals = data["vals"]
print(f"Output: {n} pairs")
print(f"  rows dtype: {data['rows'].dtype}")
print(f"  vals: [{vals.min():.4f}, {vals.max():.4f}] mean|v|={np.abs(vals).mean():.4f}")
print(f"  |v|>0.5: {(np.abs(vals) > 0.5).sum()} | |v|>0.8: {(np.abs(vals) > 0.8).sum()}")
print(f"  unique cids: {len(set(data['rows']).union(set(data['cols'])))}")
print(f"  file size: {os.path.getsize(output_path)/1e6:.1f} MB")

## 8. Download or copy result

On Colab: saved to Drive (persists after disconnect).
On DataSphere: download from file browser or use the cell below.

In [ ]:
import shutil
zip_path = output_path.replace(".npz", ".zip")
shutil.make_archive(zip_path.replace(".zip", ""), "zip", os.path.dirname(output_path),
                    os.path.basename(output_path))
print(f"File ready: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_GPU'))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print(f"\nOutput: {output_path}")

## 9. Cleanup GPU memory

In [ ]:
del model
torch.cuda.empty_cache()
print("GPU memory freed")